# MecâniQA - Encontro 6 (16/09/2026)

Feature engineering temporal: lags e janela rolante, sem vazamento de futuro.

**Decisões do brainstorm:** Lag 1 (persistência / Naive), Lag 7 (ciclo semanal) e média móvel de 7 dias só com o passado. Linhas do topo sem histórico completo saem com `dropna`.

In [ ]:
import pandas as pd

## Base diária

Interpolamos os buracos de valor (não de calendário) para o `shift` contar dias de verdade.

In [ ]:
df = pd.read_csv("data/mecaniqa_dataset.csv")
df["Data"] = pd.to_datetime(df["Data"])
df = df.set_index("Data").sort_index()

df["Trocas_Oleo"] = df["Trocas_Oleo"].interpolate(method="time")
df["Manutencao_Motor"] = df["Manutencao_Motor"].interpolate(method="time")

## Lags

- `lag_1`: ontem (mesmo sinal do Naive).
- `lag_7`: mesmo dia da semana passada (sazonalidade de 7 dias).

In [ ]:
df["lag_1"] = df["Trocas_Oleo"].shift(1)
df["lag_7"] = df["Trocas_Oleo"].shift(7)

## Janela rolante (sem data leakage)

`shift(1)` antes do `rolling`: a média do dia t usa só t-1 até t-7. O valor de hoje não entra na feature.

In [ ]:
df["mm_7"] = df["Trocas_Oleo"].shift(1).rolling(window=7).mean()

## Nulos do topo

As primeiras linhas não têm 7 dias de passado. Removemos; não preenchemos com 0 nem com a média (isso distorceria o treino).

In [ ]:
print("NaNs antes do corte:")
print(df[["lag_1", "lag_7", "mm_7"]].isna().sum())

df = df.dropna()

print("\nShape depois do dropna:", df.shape)
print("Primeiro dia útil para treino:", df.index.min().date())

## QA (critério de aceite)

`.head(15)`: features preenchidas e alinhadas no tempo. Conferência: `lag_1` = ontem, `lag_7` = 7 dias atrás, `mm_7` = média dos 7 dias anteriores (sem o dia atual).

In [ ]:
qa = df[["Trocas_Oleo", "lag_1", "lag_7", "mm_7"]].copy()
print(qa.head(15).to_string())

assert qa.head(15).isna().sum().sum() == 0, "head(15) ainda tem NaN"

serie_cheia = pd.read_csv("data/mecaniqa_dataset.csv")
serie_cheia["Data"] = pd.to_datetime(serie_cheia["Data"])
serie_cheia = serie_cheia.set_index("Data").sort_index()
serie_cheia["Trocas_Oleo"] = serie_cheia["Trocas_Oleo"].interpolate(method="time")

dia = qa.index[0]
assert df.loc[dia, "lag_1"] == serie_cheia.loc[dia - pd.Timedelta(days=1), "Trocas_Oleo"]
assert df.loc[dia, "lag_7"] == serie_cheia.loc[dia - pd.Timedelta(days=7), "Trocas_Oleo"]

passado = serie_cheia.loc[dia - pd.Timedelta(days=7) : dia - pd.Timedelta(days=1), "Trocas_Oleo"]
com_hoje = serie_cheia.loc[dia - pd.Timedelta(days=6) : dia, "Trocas_Oleo"]
assert abs(df.loc[dia, "mm_7"] - passado.mean()) < 1e-9
assert abs(df.loc[dia, "mm_7"] - com_hoje.mean()) > 1e-9

print("QA ok: sem NaN no head(15); lag_1, lag_7 e mm_7 alinhados ao passado.")

## Dataset enriquecido (entrega da Sprint 2)

In [ ]:
saida = "data/mecaniqa_features.csv"
df.to_csv(saida)
print(f"Salvo {saida} com colunas: {list(df.columns)}")